In [16]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
# Create a SparkSession
# .appName() sets a name for your application
# .getOrCreate() will create a new SparkSession or reuse an existing one
spark = SparkSession.builder \
    .appName("MyPySparkApplication") \
    .config("spark.driver.extraClassPath", "/opt/bitnami/spark/jars/postgresql-42.6.0.jar") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .getOrCreate()

In [3]:
# Verify that the SparkSession is running
print(spark.sparkContext.appName)
print(spark)

MyPySparkApplication


In [4]:
sc = spark.sparkContext

In [5]:
sc.version

'3.5.0'

In [6]:
sc.pythonVer

'3.11'

In [7]:
sc.master

'local[*]'

In [8]:
from pyspark.sql.types import *

In [9]:
# Configuration and example to read a CSV from MinIO (S3-compatible) using s3a.
# Replace the placeholders below with your MinIO credentials, endpoint, bucket and object path.
import os

MINIO_ACCESS_KEY = os.environ.get('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.environ.get('MINIO_SECRET_KEY', 'minioadmin')
MINIO_ENDPOINT = os.environ.get('MINIO_ENDPOINT', 'http://localhost:9000')  # include http:// and port

# Configure Hadoop S3A settings on the active SparkSession
hconf = spark._jsc.hadoopConfiguration()
hconf.set('fs.s3a.access.key', MINIO_ACCESS_KEY)
hconf.set('fs.s3a.secret.key', MINIO_SECRET_KEY)
hconf.set('fs.s3a.endpoint', MINIO_ENDPOINT)
hconf.set('fs.s3a.path.style.access', 'true')
hconf.set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
hconf.set('com.amazonaws.services.s3.enableV4', 'true')
# If your MinIO endpoint is not using TLS (https), disable ssl; set to 'true' for TLS endpoints
hconf.set('fs.s3a.connection.ssl.enabled', 'false')

print('Configured S3A endpoint:', MINIO_ENDPOINT)

Configured S3A endpoint: minio:9000


In [10]:
data_schema = StructType([
    StructField("No", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Article_title", StringType(), True),
    StructField("Stock_symbol", StringType(), True),
    StructField("Url", StringType(), True),
    StructField("Publisher", StringType(), True),
    StructField("Author", StringType(), True),
    StructField("Article", StringType(), True),
    StructField("Lsa_summary", StringType(), True),
    StructField("Luhn_summary", StringType(), True),
    StructField("Textrank_summary", StringType(), True),
    StructField("Lexrank_summary", StringType(), True)
])

In [13]:
bucket = 'fnspid-bucket'  # <-- replace with your bucket name
object_path = 'raw/stock_news/nasdaq_exteral_data.csv'  # <-- replace with your object path
s3a_path = f's3a://{bucket}/{object_path}'

In [26]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(data_schema) \
    .load(s3a_path)

In [27]:
df.show(1)

+---+--------------------+--------------------+------------+--------------------+---------+------+--------------------+--------------------+--------------------+--------------------+--------------------+
| No|                Date|       Article_title|Stock_symbol|                 Url|Publisher|Author|             Article|         Lsa_summary|        Luhn_summary|    Textrank_summary|     Lexrank_summary|
+---+--------------------+--------------------+------------+--------------------+---------+------+--------------------+--------------------+--------------------+--------------------+--------------------+
|0.0|2023-12-16 23:00:...|Interesting A Put...|           A|https://www.nasda...|     NULL|  NULL|Investors in Agil...|Because the $125....|The current analy...|Below is a chart ...|At Stock Options ...|
+---+--------------------+--------------------+------------+--------------------+---------+------+--------------------+--------------------+--------------------+--------------------+--

In [41]:
df = df.drop(F.col('Publisher'), F.col('Author'), F.col('Luhn_summary'), F.col('Textrank_summary'), F.col('Lexrank_summary'))

In [55]:
df.select(F.col('Stock_symbol')).distinct().count()

8553